# One-head nanoGPT baseline: performance and spectra

This is the source report notebook for the frozen AdamW and MuonClip campaign prepared on 2026-08-22. It does **not** train models or select checkpoints. It invokes the campaign report builder against already-completed run artifacts, then displays its Markdown summary, aggregate tables, and plots.

Execute it through Papermill with explicit `RESULTS_ROOT` and `OUTPUT_ROOT` parameters. Set `REQUIRE_COMPLETE=True` for the preregistered 2 x 5 report.

In [ ]:
RESULTS_ROOT = ""
OUTPUT_ROOT = ""
REQUIRE_COMPLETE = True

In [ ]:
from pathlib import Path
import subprocess
import sys

import pandas as pd
from IPython.display import Image, Markdown, display

EXPERIMENT_ID = "nanogpt_one_head_2026_08_21_baseline"


def locate_experiment_root() -> Path:
    for anchor in (Path.cwd(), *Path.cwd().parents):
        if (
            anchor.name == EXPERIMENT_ID
            and (anchor / "scripts" / "build_report.py").is_file()
        ):
            return anchor.resolve()
        candidate = (
            anchor / "baseline" / "experiments" / EXPERIMENT_ID
        )
        if (candidate / "scripts" / "build_report.py").is_file():
            return candidate.resolve()
    raise FileNotFoundError(
        f"Could not locate {EXPERIMENT_ID}; execute from the repository "
        "or the dated experiment directory."
    )


def required_absolute_path(value: str, parameter: str) -> Path:
    if not str(value).strip():
        raise ValueError(f"Papermill parameter {parameter} is required")
    path = Path(value)
    if not path.is_absolute():
        raise ValueError(f"{parameter} must be an absolute path: {path}")
    return path.resolve()


EXPERIMENT_PATH = locate_experiment_root()
RESULTS_PATH = required_absolute_path(RESULTS_ROOT, "RESULTS_ROOT")
OUTPUT_PATH = required_absolute_path(OUTPUT_ROOT, "OUTPUT_ROOT")
if not RESULTS_PATH.is_dir():
    raise FileNotFoundError(f"Results directory does not exist: {RESULTS_PATH}")
print(f"Experiment: {EXPERIMENT_PATH}")
print(f"Results:    {RESULTS_PATH}")
print(f"Report:     {OUTPUT_PATH}")

In [ ]:
report_command = [
    sys.executable,
    str(EXPERIMENT_PATH / "scripts" / "build_report.py"),
    "--results-root",
    str(RESULTS_PATH),
    "--output-root",
    str(OUTPUT_PATH),
]
if REQUIRE_COMPLETE:
    report_command.append("--require-complete")
else:
    report_command.append("--allow-incomplete")
completed = subprocess.run(
    report_command,
    check=True,
    capture_output=True,
    text=True,
)
if completed.stdout:
    print(completed.stdout.rstrip())
if completed.stderr:
    print(completed.stderr.rstrip(), file=sys.stderr)

In [ ]:
summary_paths = sorted(OUTPUT_PATH.rglob("SUMMARY.md"))
if not summary_paths:
    raise FileNotFoundError(f"Report builder wrote no SUMMARY.md below {OUTPUT_PATH}")
for summary_path in summary_paths:
    display(Markdown(summary_path.read_text(encoding="utf-8")))

In [ ]:
csv_paths = sorted(OUTPUT_PATH.rglob("*.csv"))
if not csv_paths:
    raise FileNotFoundError(f"Report builder wrote no CSV tables below {OUTPUT_PATH}")
for csv_path in csv_paths:
    label = csv_path.relative_to(OUTPUT_PATH)
    display(Markdown(f"### `{label}`"))
    display(pd.read_csv(csv_path).head(20))

In [ ]:
png_paths = sorted(OUTPUT_PATH.rglob("*.png"))
if not png_paths:
    raise FileNotFoundError(f"Report builder wrote no PNG plots below {OUTPUT_PATH}")
for png_path in png_paths:
    label = png_path.relative_to(OUTPUT_PATH)
    display(Markdown(f"### `{label}`"))
    display(Image(filename=str(png_path)))